# Select background subtracted files: cell ch

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.image as mpimg
import sys
import shutil
src_path = str(Path.cwd().parent)
if src_path not in sys.path:
    sys.path.append(src_path)
from microscopy_analysis.d01_init_proc import vis_and_rescale
from microscopy_analysis.d00_utils import dirnames as dn
from microscopy_analysis.d00_utils import utilities as utils
from microscopy_analysis.d01_init_proc.subtractbg import subtract_bg

## Load background subtract dataframe

In [ ]:
ch_content = 'cell'
ch = 1

In [ ]:
bgsub_df_path = Path(input())

In [ ]:
bgsub_df = pd.read_csv(bgsub_df_path)
bgsub_df['basename'] = [name[:(np.char.find(name, 'aligned') + len('aligned'))] for name in bgsub_df['input image name']]
bgsub_df.head()

## Create table with mask information

In [ ]:
proc_dirpath = utils.get_proc_dirpath(bgsub_df_path)
polygonmasks_dirpath = proc_dirpath / dn.masks_dirname / dn.polygon_ROI_dirname
assert polygonmasks_dirpath.is_dir()

In [ ]:
tables_dirpath = proc_dirpath / dn.tables_dirname
masks_df_path = tables_dirpath / 'masks_df.csv'

if masks_df_path.is_file():
    masks_df = pd.read_csv(masks_df_path)
else:
    polygonmasks_dirpath = proc_dirpath / dn.masks_dirname / dn.polygon_ROI_dirname
    assert polygonmasks_dirpath.is_dir()

    masknames = [mask.name for mask in polygonmasks_dirpath.glob('*.png')]
    masknames.sort()

    masks_df = pd.DataFrame()
    masks_df['mask name'] = masknames
    masks_df['mask dirpath'] = polygonmasks_dirpath
    splits = masks_df['mask name'].str.split('_')

    # may need to modify based on names
    masks_df['basename'] = [name[:len(name)-9] for name in masks_df['mask name']]
    masks_df['wellID'] = splits.str[2]
    masks_df['scene'] = (splits.str[3]).str.split('-').str[0]
    masks_df['ROI'] = [name[len(name)-8:len(name)-4] for name in masks_df['mask name']]
    masks_df.head()

    masks_df_path = bgsub_df_path.parent / 'masks_df.csv'
    utils.safe_save_csv(masks_df, masks_df_path)

In [ ]:
# may need to modify based on names
masks_df['basename'] = [name[:len(name)-9] for name in masks_df['mask name']]
masks_df['wellID'] = splits.str[2]
masks_df['scene'] = (splits.str[3]).str.split('-').str[0]
masks_df['ROI'] = [name[len(name)-8:len(name)-4] for name in masks_df['mask name']]
masks_df.head()

In [ ]:
masks_df.head()

In [ ]:
bgsub_masks_df_path = bgsub_df_path.parent / f'bgsub_{ch_content}_masks.csv'

bgsub_cell_masks_df = pd.merge(masks_df, bgsub_df, on='basename', how='left')
assert (len(masks_df) == (len(bgsub_cell_masks_df)))
bgsub_cell_masks_df.head()

In [ ]:
cols = np.array(bgsub_cell_masks_df.columns.to_list())

for i, col in enumerate(cols):
    print(f'{i}: {col}')

In [ ]:
new_col_order = ['img idx', 'basename', 'wellID', 'scene', 'ROI', 'selected param', 'BG subtract?', 'ch_to_process', 'outlier_percentiles', 'rescale_perc_grayval', 'sigmas_smoothing', '# prev tested params', 'mask name', 'mask dirpath', 'input image name', 'input dirname', 'output dirpath']

bgsub_cell_masks_df = bgsub_cell_masks_df[new_col_order]
bgsub_cell_masks_df.head()

In [ ]:
utils.safe_save_csv(bgsub_cell_masks_df, bgsub_masks_df_path)

## Visualize background subtracted figures

In [ ]:
bgsub_dirpath = proc_dirpath / dn.bg_sub_dirname / f'caax_cell_stack_binch{ch}_init'
assert bgsub_dirpath.is_dir()

fig_dirpath = bgsub_dirpath / dn.figs_dirname
figpaths = list(fig_dirpath.glob('*.png'))
figpaths.sort()
num_figs = len(figpaths)
print(num_figs)

In [ ]:
subset=None

#### Note: open "bgsub" file and use visualized figures to fill in the "selected params" column

In [ ]:
vis_and_rescale.show_figs(fig_dirpath, subset=subset)

## Re-run background subtraction (optional)

In [ ]:
bgsub_masks_df = pd.read_csv(bgsub_masks_df_path)
bgsub_masks_df.head()

In [ ]:
input_dirpath = utils.get_proc_dirpath(polygonmasks_dirpath) / 'caax_cell_stack'
#input_dirpath = utils.get_proc_dirpath(polygonmasks_dirpath) / dn.stack_dirname

subtract_bg(input_dirpath, bgsub_masks_df_path)

## Update bgsub masks dataframe with selected images

In [ ]:
bgsub_masks_df_path

In [ ]:
bgsub_masks_df = pd.read_csv(bgsub_masks_df_path)
bgsub_masks_df.head()

In [ ]:
bgsub_masks_df[f'bgsub {ch_content} img']  = bgsub_masks_df['input image name'].str.split('.').str[0] + '_p' + bgsub_masks_df['selected param'].astype('string') + '.ome.tif'
bgsub_masks_df.loc[bgsub_masks_df['selected param']=='omit', f'bgsub {ch_content} img'] = 'NA'

bgsub_masks_df.head()

In [ ]:
sel_bgsub_dirpath = bgsub_dirpath.parent / (bgsub_dirpath.name.split('_init')[0] + '_sel')
print(sel_bgsub_dirpath)

In [ ]:
# record selected background-subtracted image dirpath
bgsub_masks_df[f'bgsub {ch_content} dirpath'] = sel_bgsub_dirpath

In [ ]:
utils.safe_save_csv(bgsub_masks_df, bgsub_masks_df_path)

# Create dataframe with masks and corresponding selected bgsub images 

In [ ]:
bgsub_masks_df.head()

In [ ]:
masks_df.head()

In [ ]:
print(f'masks_df rows: {len(masks_df)}')
print(f'bgsub {ch_content} rows: {len(masks_df)}')

sel_bgsub_df = pd.merge(masks_df, bgsub_masks_df[['mask name', f'bgsub {ch_content} img', f'bgsub {ch_content} dirpath']])

print(f'sel_bgsub_df rows: {len(sel_bgsub_df)}')
sel_bgsub_df.head()

In [ ]:
tables_dirpath = utils.get_proc_dirpath(bgsub_df_path) / dn.tables_dirname
sel_bgsub_df_path = tables_dirpath / 'bgsub_selected_bymask.csv'
utils.safe_save_csv(sel_bgsub_df, sel_bgsub_df_path)

## Create thresholds table with only selected images

In [ ]:
bgsub_thresh_path = str(bgsub_df_path).split('.csv')[0] + '_thresh.csv'
bgsub_thresh_df = pd.read_csv(bgsub_thresh_path)
bgsub_thresh_df.head()

In [ ]:
bgsub_thresh_df['image name'] = bgsub_thresh_df['image name'] + '.ome.tif'

In [ ]:
sel_imgs_uniq = sel_bgsub_df[f'bgsub {ch_content} img'].unique()
bgsub_thresh_sel_df = bgsub_thresh_df[bgsub_thresh_df['image name'].str.contains('|'.join(sel_imgs_uniq))]
bgsub_thresh_sel_df.head()

In [ ]:
print(len(bgsub_thresh_sel_df))

In [ ]:
bgsub_thresh_sel_path = Path(str(bgsub_df_path).split('.csv')[0] + '_thresh_selected.csv')
print(bgsub_thresh_sel_path.name)

In [ ]:
utils.safe_save_csv(bgsub_thresh_sel_df, bgsub_thresh_sel_path)

## Move selected images

In [ ]:
assert bgsub_dirpath.is_dir()
print(f'bgsub dirpath: {bgsub_dirpath}\n')
print(f'sel bgsub dirpath: {sel_bgsub_dirpath}\n')

sel_fig_dirpath = sel_bgsub_dirpath / dn.figs_dirname
print(f'sel bgsub fig dirpath: {sel_fig_dirpath}')

In [ ]:
sel_fig_dirpath.mkdir(parents=True, exist_ok=True)

In [ ]:
imgs_moved = 0
figs_moved = 0

for img in sel_imgs_uniq:
    # move bg subtracted files
    imgpath = bgsub_dirpath / img

    if imgpath.is_file():
        shutil.copy(imgpath, sel_bgsub_dirpath / img)
        imgs_moved = imgs_moved + 1
    else:
        print(f'{img} not found.') 
    
    # # move bg subtracted figures
    # figname = img.replace('.ome.tif', '.png')
    # figpath = fig_dirpath / figname
    # if figpath.is_file():
    #     shutil.copy(figpath, sel_fig_dirpath / figname)
    #     figs_moved = figs_moved + 1
    # else:
    #     print(f'{figname} not found.') 
        
print(f'Copied {imgs_moved} images and {figs_moved} figures.')